In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base = '/content/drive/MyDrive/chicken-ai'
print(os.listdir(base))

['Noise', 'Healthy', 'Unhealthy', 'yamnet_embeddings.npy', 'mels.npy', 'features.csv', 'final_svm.joblib']


In [ ]:
!pip install librosa

In [ ]:
import os, glob
from collections import Counter

base = '/content/drive/MyDrive/chicken-ai'
files = glob.glob(base + '/**/*.wav', recursive=True)
print("Total wav files:", len(files))

counts = Counter(os.path.basename(os.path.dirname(f)) for f in files)
print(counts)
print(files[:3])

Total wav files: 346
Counter({'Healthy': 139, 'Unhealthy': 121, 'Noise': 86})
['/content/drive/MyDrive/chicken-ai/Noise/1.wav', '/content/drive/MyDrive/chicken-ai/Noise/12.wav', '/content/drive/MyDrive/chicken-ai/Noise/13.wav']


In [ ]:
import librosa
import IPython.display as ipd

for cls in ['Healthy', 'Noise', 'Unhealthy']:
    f = [x for x in files if '/' + cls + '/' in x][0]
    y, sr = librosa.load(f, sr=16000)
    print(cls, '|', f.split('/')[-1], '| seconds:', round(len(y)/sr, 1))
    ipd.display(ipd.Audio(y, rate=sr))

Healthy | 103.wav | seconds: 6.8


Noise | 1.wav | seconds: 2.1


Unhealthy | 100.wav | seconds: 1.2


In [ ]:
import soundfile as sf
import pandas as pd

rows = []
for f in files:
    info = sf.info(f)
    cls = f.split('/')[-2]
    rows.append([cls, f.split('/')[-1], info.duration, info.samplerate])

lens = pd.DataFrame(rows, columns=['class', 'file', 'seconds', 'sr'])
print(lens.groupby('class')['seconds'].describe().round(1))
print(lens['sr'].value_counts())

           count  mean   std  min   25%   50%   75%    max
class                                                     
Healthy    139.0  33.5  80.5  6.1  12.9  20.8  32.8  906.2
Noise       86.0  37.8  25.1  0.6  18.7  34.4  54.3  104.0
Unhealthy  121.0  28.7  26.6  0.5   4.3  22.6  42.2  134.0
sr
48000    346
Name: count, dtype: int64


In [ ]:
import numpy as np, pandas as pd, librosa, random
random.seed(42)

SR = 16000
CLIP = SR          # 1 second
MAX_PER_FILE = 10  # same cap for every class
rows = []

def feats(seg):
    mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=40)
    return list(np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]))

for f in files:
    cls = f.split('/')[-2]
    name = cls + '_' + f.split('/')[-1]   # unique file name
    y, _ = librosa.load(f, sr=SR, duration=60)

    if len(y) < CLIP:
        y = np.pad(y, (0, CLIP - len(y)))

    starts = list(range(0, len(y) - CLIP + 1, CLIP))
    if len(starts) < 4:  # very short file: use overlapping windows
        starts = list(range(0, len(y) - CLIP + 1, CLIP // 4))
    random.shuffle(starts)
    starts = starts[:MAX_PER_FILE]

    for s in starts:
        rows.append([name, cls] + feats(y[s:s+CLIP]))

cols = ['file', 'label'] + [f'f{i}' for i in range(80)]
df = pd.DataFrame(rows, columns=cols)
df.to_csv(base + '/features.csv', index=False)

print(df.shape)
print(df['label'].value_counts())
print(df.groupby('label')['file'].nunique())

(3122, 82)
label
Healthy      1359
Unhealthy     970
Noise         793
Name: count, dtype: int64
label
Healthy      139
Noise         86
Unhealthy    121
Name: file, dtype: int64


In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, confusion_matrix

X = df.drop(columns=['file', 'label']).values
y = df['label'].values
groups = df['file'].values

gkf = GroupKFold(n_splits=5)
all_true, all_pred = [], []

for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
    model = RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
    model.fit(X[tr], y[tr])
    pred = model.predict(X[te])
    print(f"Fold {fold}: accuracy = {(pred == y[te]).mean():.3f}")
    all_true += list(y[te])
    all_pred += list(pred)

print()
print(classification_report(all_true, all_pred, digits=3))
labels = ['Healthy', 'Noise', 'Unhealthy']
print(pd.DataFrame(confusion_matrix(all_true, all_pred, labels=labels),
                   index=labels, columns=labels))

Fold 1: accuracy = 0.890
Fold 2: accuracy = 0.958
Fold 3: accuracy = 0.947
Fold 4: accuracy = 0.965
Fold 5: accuracy = 0.929

              precision    recall  f1-score   support

     Healthy      0.899     0.972     0.934      1359
       Noise      0.967     0.837     0.897       793
   Unhealthy      0.976     0.972     0.974       970

    accuracy                          0.938      3122
   macro avg      0.947     0.927     0.935      3122
weighted avg      0.940     0.938     0.937      3122

           Healthy  Noise  Unhealthy
Healthy       1321     22         16
Noise          122    664          7
Unhealthy       26      1        943


In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GroupKFold
import numpy as np

gkf = GroupKFold(n_splits=5)

# 1) Sanity check: shuffle labels at the FILE level. Accuracy should drop to about 0.33-0.45.
files_u = df.drop_duplicates('file')[['file', 'label']].copy()
rng = np.random.RandomState(0)
files_u['fake'] = rng.permutation(files_u['label'].values)
fake_map = dict(zip(files_u['file'], files_u['fake']))
y_fake = df['file'].map(fake_map).values

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42, n_jobs=-1)
s = cross_val_score(rf, X, y_fake, groups=groups, cv=gkf)
print("Shuffled-label accuracy:", s.mean().round(3))

# 2) Compare models on the real labels
svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, class_weight='balanced'))
for name, m in [('RandomForest', rf), ('SVM', svm)]:
    s = cross_val_score(m, X, y, groups=groups, cv=gkf)
    print(name, "accuracy:", s.mean().round(3), "+/-", s.std().round(3))

Shuffled-label accuracy: 0.329
RandomForest accuracy: 0.937 +/- 0.027
SVM accuracy: 0.942 +/- 0.018


In [ ]:
import numpy as np, librosa, random
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

random.seed(42)
SR = 16000
CLIP = SR
MAX_PER_FILE = 10
emb_rows, names, labels_list = [], [], []

for f in files:
    cls = f.split('/')[-2]
    name = cls + '_' + f.split('/')[-1]
    y_, _ = librosa.load(f, sr=SR, duration=60)
    if len(y_) < CLIP:
        y_ = np.pad(y_, (0, CLIP - len(y_)))
    starts = list(range(0, len(y_) - CLIP + 1, CLIP))
    if len(starts) < 4:
        starts = list(range(0, len(y_) - CLIP + 1, CLIP // 4))
    random.shuffle(starts)
    starts = starts[:MAX_PER_FILE]
    for s in starts:
        seg = y_[s:s+CLIP].astype(np.float32)
        _, emb, _ = yamnet(seg)
        emb_rows.append(emb.numpy().mean(axis=0))
        names.append(name)
        labels_list.append(cls)

E = np.array(emb_rows)
np.save(base + '/yamnet_embeddings.npy', E)
print(E.shape)
print("Rows match table:", list(df['file']) == names)

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


(3122, 1024)
Rows match table: True


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

y = df['label'].values
groups = df['file'].values
gkf = GroupKFold(n_splits=5)

models = {
    'YAMNet + LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'YAMNet + SVM': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, class_weight='balanced')),
}

for name, m in models.items():
    pred = cross_val_predict(m, E, y, groups=groups, cv=gkf)
    print("=====", name, "accuracy:", round((pred == y).mean(), 3))
    print(classification_report(y, pred, digits=3))
    labs = ['Healthy', 'Noise', 'Unhealthy']
    print(pd.DataFrame(confusion_matrix(y, pred, labels=labs), index=labs, columns=labs))

===== YAMNet + LogReg accuracy: 0.772
              precision    recall  f1-score   support

     Healthy      0.782     0.767     0.774      1359
       Noise      0.626     0.643     0.634       793
   Unhealthy      0.880     0.885     0.882       970

    accuracy                          0.772      3122
   macro avg      0.763     0.765     0.764      3122
weighted avg      0.773     0.772     0.772      3122

           Healthy  Noise  Unhealthy
Healthy       1042    255         62
Noise          228    510         55
Unhealthy       62     50        858
===== YAMNet + SVM accuracy: 0.815
              precision    recall  f1-score   support

     Healthy      0.797     0.844     0.820      1359
       Noise      0.717     0.652     0.683       793
   Unhealthy      0.917     0.908     0.912       970

    accuracy                          0.815      3122
   macro avg      0.810     0.801     0.805      3122
weighted avg      0.814     0.815     0.814      3122

           Health

In [ ]:
import numpy as np, librosa, random
random.seed(42)

SR = 16000
CLIP = SR
MAX_PER_FILE = 10
mels, names = [], []

for f in files:
    cls = f.split('/')[-2]
    name = cls + '_' + f.split('/')[-1]
    y_, _ = librosa.load(f, sr=SR, duration=60)
    if len(y_) < CLIP:
        y_ = np.pad(y_, (0, CLIP - len(y_)))
    starts = list(range(0, len(y_) - CLIP + 1, CLIP))
    if len(starts) < 4:
        starts = list(range(0, len(y_) - CLIP + 1, CLIP // 4))
    random.shuffle(starts)
    starts = starts[:MAX_PER_FILE]
    for s in starts:
        seg = y_[s:s+CLIP]
        m = librosa.feature.melspectrogram(y=seg, sr=SR, n_fft=1024,
                                           hop_length=256, n_mels=64)
        m = librosa.power_to_db(m, ref=1.0)
        m = (m - m.mean()) / (m.std() + 1e-6)   # per-clip normalization
        mels.append(m)
        names.append(name)

M = np.array(mels, dtype='float32')[..., np.newaxis]
np.save(base + '/mels.npy', M)
print(M.shape)
print("Rows match table:", names == list(df['file']))

(3122, 64, 63, 1)
Rows match table: True


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

classes = ['Healthy', 'Noise', 'Unhealthy']
yi = np.array([classes.index(l) for l in df['label']])
groups = df['file'].values

def build():
    m = models.Sequential([
        layers.Input(shape=M.shape[1:]),
        layers.Conv2D(16, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax'),
    ])
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    return m

gkf = GroupKFold(n_splits=5)
all_true, all_pred = [], []

for fold, (tr, te) in enumerate(gkf.split(M, yi, groups), 1):
    tf.keras.backend.clear_session()
    w = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=yi[tr])
    model = build()
    model.fit(M[tr], yi[tr], epochs=20, batch_size=32,
              class_weight=dict(enumerate(w)), verbose=0)
    pred = model.predict(M[te], verbose=0).argmax(axis=1)
    print(f"Fold {fold}: accuracy = {(pred == yi[te]).mean():.3f}")
    all_true += list(yi[te])
    all_pred += list(pred)

print()
print(classification_report(all_true, all_pred, target_names=classes, digits=3))
print(pd.DataFrame(confusion_matrix(all_true, all_pred),
                   index=classes, columns=classes))

Fold 1: accuracy = 0.846
Fold 2: accuracy = 0.842
Fold 3: accuracy = 0.889
Fold 4: accuracy = 0.912
Fold 5: accuracy = 0.902

              precision    recall  f1-score   support

     Healthy      0.847     0.914     0.879      1359
       Noise      0.843     0.719     0.776       793
   Unhealthy      0.950     0.959     0.954       970

    accuracy                          0.878      3122
   macro avg      0.880     0.864     0.870      3122
weighted avg      0.878     0.878     0.876      3122

           Healthy  Noise  Unhealthy
Healthy       1242     94         23
Noise          197    570         26
Unhealthy       28     12        930


In [ ]:
import joblib
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['file', 'label']).values
y = df['label'].values

final = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=10, class_weight='balanced', probability=True, random_state=42))
final.fit(X, y)
joblib.dump(final, base + '/final_svm.joblib')
print("Saved. Classes:", final.classes_)

Saved. Classes: ['Healthy' 'Noise' 'Unhealthy']


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr, numpy as np, librosa

SR = 16000
CLIP = SR
model = joblib.load(base + '/final_svm.joblib')
classes = list(model.classes_)

def feats(seg):
    mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=40)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

def predict(path):
    if path is None:
        return "Please record or upload a sound first."
    y_, _ = librosa.load(path, sr=SR, duration=60)
    if len(y_) < CLIP:
        y_ = np.pad(y_, (0, CLIP - len(y_)))
    starts = list(range(0, len(y_) - CLIP + 1, CLIP // 2))
    X_new = np.array([feats(y_[s:s+CLIP]) for s in starts])
    p = model.predict_proba(X_new).mean(axis=0)
    out = {c: float(v) for c, v in zip(classes, p)}
    return out

demo = gr.Interface(
    fn=predict,
    inputs=gr.Audio(type="filepath", label="Record or upload chicken sound"),
    outputs=gr.Label(num_top_classes=3, label="Result"),
    title="Poultry Sound Health Checker",
    description="Upload or record a few seconds of poultry sound. The model estimates Healthy, Unhealthy, or Noise. Research demo trained on one public dataset.",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0548e62c9f8c5b846e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
